In [ ]:
import pandas as pd

# Creacion de la base de datos

In [ ]:
comprobantes = pd.read_excel('Comprobantes detallados 2025 SIIGO.xlsx', header=7)
comprobantes.head(10)

In [ ]:
# Extraer código de comprobante de las filas encabezado y propagarlo
comprobantes['Comprobante'] = comprobantes['Secuencia'].where(
    comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')
).ffill()

# Extraer solo el código (ej: 'A-272' de 'Comprobante: A-272')
comprobantes['Comprobante'] = comprobantes['Comprobante'].str.replace('Comprobante: ', '', regex=False)

# Eliminar las filas encabezado de comprobante
comprobantes = comprobantes[~comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')].reset_index(drop=True)

# Convertir columnas a entero (Int64 soporta NaN)
comprobantes['Código contable'] = comprobantes['Código contable'].astype('Int64')
comprobantes['Identificación'] = comprobantes['Identificación'].astype('Int64')
comprobantes['Sucursal'] = comprobantes['Sucursal'].astype('Int64')

# Crear columna Movimiento
comprobantes['Movimiento'] = comprobantes.apply(
    lambda r: r['Crédito'] if r['Débito'] == 0 else -r['Débito'], axis=1
)

# Mover Comprobante al inicio
cols = ['Comprobante'] + [c for c in comprobantes.columns if c != 'Comprobante']
comprobantes = comprobantes[cols]

comprobantes.head(20)

In [ ]:
productos = pd.read_excel('Listado de productos _ Servicios .xlsx', header=6)

# Filtrar solo filas válidas (Producto o Servicio)
productos = productos[productos['Tipo'].isin(['Producto', 'Servicio'])].copy().reset_index(drop=True)

# Renombrar para consistencia con el resto del notebook
productos = productos.rename(columns={'Código': 'Código producto', 'Nombre': 'Nombre producto'})

productos.head(10)

In [ ]:
# Extraer código de producto desde Detalle (ej: 'Prod: NX10011765 Cant: 1.00')
comprobantes['Código producto'] = comprobantes['Detalle'].str.extract(r'Prod:\s*(\S+)')

# Merge con productos para traer Categoría
comprobantes = comprobantes.merge(
    productos[['Código producto', 'Categoría']],
    on='Código producto', how='left'
)

comprobantes.head(20)

In [ ]:
import re

sin_match = (
    comprobantes['Código producto'].isna() &
    comprobantes['Detalle'].str.contains(r'Prod(?:ucto)?:', na=False)
)
print(f'Sin match inicial: {sin_match.sum()}')

# --- Paso 1: Match exacto por nombre de producto ---
lookup_exacto = productos.drop_duplicates('Nombre producto').set_index('Nombre producto')
desc_exacta = comprobantes.loc[sin_match, 'Descripción']
matched_exacto = desc_exacta.map(lookup_exacto['Código producto'])

for col in ['Código producto', 'Categoría']:
    idx = matched_exacto.index[matched_exacto.notna()]
    comprobantes.loc[idx, col] = desc_exacta[matched_exacto.notna()].map(lookup_exacto[col]).values

sin_match = comprobantes['Código producto'].isna() & comprobantes['Detalle'].str.contains(r'Prod(?:ucto)?:', na=False)
print(f'Sin match después de exacto: {sin_match.sum()}')

# --- Paso 2: Match normalizado (quitar marcas al final) ---
def normalizar_desc(s):
    s = str(s).replace('"', '').strip()
    s = re.sub(r'\s+M\.?\s*GEN(?:E(?:RICA)?)?$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+M\.\s*\S+$', '', s)
    return s.rstrip('. ')

productos['nombre_norm'] = productos['Nombre producto'].apply(normalizar_desc)
lookup_norm = productos.drop_duplicates('nombre_norm').set_index('nombre_norm')
desc_norm = comprobantes.loc[sin_match, 'Descripción'].apply(normalizar_desc)
matched_norm = desc_norm.map(lookup_norm['Código producto'])

for col in ['Código producto', 'Categoría']:
    idx = matched_norm.index[matched_norm.notna()]
    comprobantes.loc[idx, col] = desc_norm[matched_norm.notna()].map(lookup_norm[col]).values

sin_match_final = comprobantes['Código producto'].isna() & comprobantes['Detalle'].str.contains(r'Prod(?:ucto)?:', na=False)
print(f'Sin match final: {sin_match_final.sum()}')

if sin_match_final.any():
    restantes = sorted(comprobantes.loc[sin_match_final, 'Descripción'].unique())
    print(f"\nDescripciones aún sin match ({len(restantes)}):")
    for d in restantes: print(f"  - {d}")


In [ ]:
# Asignaciones manuales para casos que la normalización no resuelve
# Formato: {descripcion_en_comprobante: codigo_en_productos}
asignaciones_manuales = {
}

productos_idx = productos.set_index('Código producto')

for desc, codigo in asignaciones_manuales.items():
    mask = (comprobantes['Descripción'] == desc) & comprobantes['Código producto'].isna()
    if mask.any() and codigo in productos_idx.index:
        comprobantes.loc[mask, 'Código producto'] = codigo
        comprobantes.loc[mask, 'Categoría'] = productos_idx.loc[codigo, 'Categoría']
        print(f"  {desc} → {codigo} ({mask.sum()} filas)")

# Asignar código especial a los que quedan sin match
sin_match = (
    comprobantes['Código producto'].isna() &
    comprobantes['Detalle'].str.contains(r'Prod(?:ucto)?:', na=False)
)
comprobantes.loc[sin_match, 'Código producto'] = 'DESCONOCIDO'
print(f'Asignados como DESCONOCIDO: {sin_match.sum()}')


In [ ]:
productos.to_csv('productos.csv', index=False)
comprobantes.to_csv('comprobantes.csv', index=False)
print('CSVs guardados.')


# Procesamiento de datos

In [ ]:
# Filtrar facturas de venta (FV) con cuenta de ingresos (41350101)
ventas = comprobantes[
    (comprobantes['Comprobante'].str.startswith('FV')) &
    (comprobantes['Código contable'] == 41350101)
].copy()

# Extraer cantidad y fecha
ventas['Cantidad'] = ventas['Detalle'].str.extract(r'Cant:\s*([\d.]+)').astype(float)
ventas['Fecha elaboración'] = pd.to_datetime(ventas['Fecha elaboración'], dayfirst=True)
ventas['Mes'] = ventas['Fecha elaboración'].dt.to_period('M')

# Agrupar por producto
ventas_por_producto = ventas.groupby('Código producto').agg(
    Descripción=('Descripción', 'first'),
    Categoría=('Categoría', 'first'),
    Cantidad_total=('Cantidad', 'sum'),
    Venta_total=('Crédito', 'sum')
).sort_values('Venta_total', ascending=False).reset_index()

print(f'Productos vendidos: {len(ventas_por_producto)}')
print(f'Venta total: ${ventas_por_producto["Venta_total"].sum():,.0f}')
ventas_por_producto

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
fig.suptitle('Dashboard Ejecutivo de Ventas - 2025', fontsize=18, fontweight='bold', y=0.98)
plt.subplots_adjust(hspace=0.45, wspace=0.35)

# --- 1. Top 15 productos por ingreso ---
ax1 = axes[0, 0]
top15 = ventas_por_producto.head(15).copy()
top15['label'] = top15['Descripción'].str[:35]
colors1 = ['#e74c3c' if x == 'DESCONOCIDO' else '#2c3e50' for x in top15['Código producto']]
ax1.barh(range(len(top15)), top15['Venta_total'], color=colors1)
ax1.set_yticks(range(len(top15)))
ax1.set_yticklabels(top15['label'], fontsize=8)
ax1.invert_yaxis()
ax1.set_xlabel('Venta ($)')
ax1.set_title('Top 15 Productos por Ingreso', fontweight='bold')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

# --- 2. Ventas por Categoría ---
ax2 = axes[0, 1]
por_cat = ventas_por_producto.groupby('Categoría')['Venta_total'].sum().sort_values(ascending=False)
por_cat = por_cat[por_cat.index.notna()]
colors2 = ['#3498db', '#e67e22', '#2ecc71']
wedges, texts, autotexts = ax2.pie(
    por_cat, labels=por_cat.index, autopct='%1.1f%%',
    colors=colors2[:len(por_cat)], startangle=90, pctdistance=0.8
)
for t in autotexts:
    t.set_fontsize(9)
    t.set_fontweight('bold')
ax2.set_title('Ventas por Categoría', fontweight='bold')

# --- 3. Ventas por Mes ---
ax3 = axes[0, 2]
ventas_mes = ventas.groupby('Mes')['Crédito'].sum().reset_index()
ventas_mes['Mes_str'] = ventas_mes['Mes'].dt.strftime('%b %Y')
ax3.bar(range(len(ventas_mes)), ventas_mes['Crédito'], color='#3498db')
ax3.set_xticks(range(len(ventas_mes)))
ax3.set_xticklabels(ventas_mes['Mes_str'], rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('Venta ($)')
ax3.set_title('Ventas Mensuales 2025', fontweight='bold')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

# --- 4. Pareto: concentración de ventas ---
ax4 = axes[1, 0]
vpp_sorted = ventas_por_producto.sort_values('Venta_total', ascending=False).reset_index(drop=True)
cum_pct = vpp_sorted['Venta_total'].cumsum() / vpp_sorted['Venta_total'].sum() * 100
ax4.bar(range(len(vpp_sorted)), vpp_sorted['Venta_total'], color='#3498db', alpha=0.7)
ax4_twin = ax4.twinx()
ax4_twin.plot(range(len(vpp_sorted)), cum_pct, color='#e74c3c', linewidth=2)
ax4_twin.axhline(y=80, color='gray', linestyle='--', alpha=0.7)
n80 = int((cum_pct <= 80).sum())
ax4_twin.annotate(f'80% del ingreso\nen {n80} productos\n({n80/len(vpp_sorted)*100:.0f}% del catálogo)',
                  xy=(n80, 80), fontsize=9, color='#e74c3c',
                  xytext=(n80 + 20, 65), arrowprops=dict(arrowstyle='->', color='#e74c3c'))
ax4.set_xlabel('Productos (ordenados por venta)')
ax4.set_ylabel('Venta ($)')
ax4_twin.set_ylabel('% Acumulado')
ax4.set_title('Concentración de Ventas (Pareto)', fontweight='bold')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

# --- 5. Top 10 clientes ---
ax5 = axes[1, 1]
ventas_cliente = ventas.groupby('Nombre tercero')['Crédito'].sum().sort_values(ascending=False).head(10)
ventas_cliente_labels = [n[:30] for n in ventas_cliente.index]
ax5.barh(range(len(ventas_cliente)), ventas_cliente.values, color='#27ae60')
ax5.set_yticks(range(len(ventas_cliente)))
ax5.set_yticklabels(ventas_cliente_labels, fontsize=8)
ax5.invert_yaxis()
ax5.set_xlabel('Venta ($)')
ax5.set_title('Top 10 Clientes por Ingreso', fontweight='bold')
ax5.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

# --- 6. KPIs resumen ---
ax6 = axes[1, 2]
ax6.axis('off')
total_ventas = ventas_por_producto['Venta_total'].sum()
n_productos = len(ventas_por_producto)
n_clientes = ventas['Nombre tercero'].nunique()
ticket_promedio = ventas['Crédito'].mean()
top1_pct = ventas_por_producto.iloc[0]['Venta_total'] / total_ventas * 100
mes_pico = ventas_mes.loc[ventas_mes['Crédito'].idxmax(), 'Mes_str']

kpis = [
    ('Venta Total 2025', f'${total_ventas/1e6:.1f}M'),
    ('Productos Vendidos', f'{n_productos}'),
    ('Clientes Activos', f'{n_clientes}'),
    ('Ticket Promedio', f'${ticket_promedio:,.0f}'),
    ('Top 1 Producto', f'{top1_pct:.1f}% del ingreso'),
    ('80% Ingreso en', f'{n80} productos ({n80/n_productos*100:.0f}%)'),
    ('Mes Pico', mes_pico),
]

for i, (label, value) in enumerate(kpis):
    y = 0.90 - i * 0.14
    ax6.text(0.5, y, value, transform=ax6.transAxes, fontsize=15,
             fontweight='bold', ha='center', va='center', color='#2c3e50')
    ax6.text(0.5, y - 0.05, label, transform=ax6.transAxes, fontsize=9,
             ha='center', va='center', color='#7f8c8d')

ax6.set_title('KPIs Clave', fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('output.png', dpi=150, bbox_inches='tight')
plt.show()
